# Monte Carlo Greeks: Pathwise & Likelihood-Ratio Estimators

Computing option sensitivities (Greeks) by Monte Carlo, as an alternative to
bump-and-revalue. Both methods here are *unbiased* - their only error is
sampling variance - and both extend to models with no closed-form Greeks
(Heston W4, rBergomi W9), which is the entire reason to build them.

## Setup

Under $\mathbb{Q}$, GBM gives a lognormal terminal price:

$$S_T = S_0 \exp\!\Big((r - \tfrac12\sigma^2)T + \sigma\sqrt{T}\,Z\Big), \qquad Z \sim N(0,1)$$

so $\log S_T \sim N(m, s^2)$ with $m = \log S_0 + (r - \tfrac12\sigma^2)T$ and
$s = \sigma\sqrt{T}$. The price is $V = e^{-rT}\,\mathbb{E}[f(S_T)]$, and we want
$\partial V / \partial\theta$.

With $d_1 = \dfrac{\log(S_0/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}$ and
$d_2 = d_1 - \sigma\sqrt{T}$

## Two ways to move the derivative inside the expectation

**Pathwise (PW)** - fix the draw $Z$, differentiate the *payoff*:

$$\frac{\partial V}{\partial\theta} = e^{-rT}\,\mathbb{E}\!\left[f'(S_T)\,\frac{\partial S_T}{\partial\theta}\right]$$

Valid only when $\theta \mapsto f(S_T(\theta, Z))$ is Lipschitz (continuous, no
jumps). Kinks are fine (hit with probability 0); jumps are fatal.

**Likelihood-ratio (LR)** - leave the payoff alone, differentiate the *density*:

$$\frac{\partial V}{\partial\theta} = e^{-rT}\,\mathbb{E}\!\left[f(S_T)\,\frac{\partial \log p(S_T;\theta)}{\partial\theta}\right]$$

The score $\partial_\theta \log p$ asks nothing of $f$ - the payoff appears raw,
so digitals/barriers/jumps are all fine. Payoff-agnostic by construction.

> **PW differentiates the payoff** (needs smoothness, low variance).
> **LR differentiates the density** (eats any payoff, pays in variance).

In [ ]:
import sys
from scipy.stats import norm
import numpy as np
from models.mc_greeks import bsm_greeks_pw, bsm_greeks_lr

In [2]:
def analytic_delta_vega(S, K, T, r, sigma):
    """Closed-form call Delta and Vega — the validation truth."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    delta = norm.cdf(d1)
    vega = S * norm.pdf(d1) * np.sqrt(T)
    return delta, vega

## Pathwise estimators (European call)

$$\Delta_{\text{PW}} = e^{-rT}\,\mathbb{E}\!\left[\mathbf{1}_{\{S_T>K\}}\,\frac{S_T}{S_0}\right] \;\;\xrightarrow{\text{mean}}\;\; \Phi(d_1)$$

$$\mathcal{V}_{\text{PW}} = e^{-rT}\,\mathbb{E}\!\left[\mathbf{1}_{\{S_T>K\}}\,S_T\big(\sqrt{T}\,Z - \sigma T\big)\right] \;\;\xrightarrow{\text{mean}}\;\; S_0\,\varphi(d_1)\sqrt{T}$$

**Put:** flip the indicator *and* the sign on delta -
$\Delta_{\text{PW,put}} = e^{-rT}\,\mathbb{E}\!\left[-\mathbf{1}_{\{S_T<K\}}\,S_T/S_0\right] \to \Phi(d_1) - 1$.
Vega is **identical** to the call (parity $C - P = S_0 - Ke^{-rT}$ has no
$\sigma$, so $\mathcal{V}_C = \mathcal{V}_P$).

In [3]:
# --- canonical ATM test case for Pathwise ---
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_paths, seed = 1_000_000, 42

pw = bsm_greeks_pw(S, K, T, r, sigma, n_paths=n_paths, seed=seed)
delta_true, vega_true = analytic_delta_vega(S, K, T, r, sigma)

for name, truth in [("delta", delta_true), ("vega", vega_true)]:
    est, se = pw[name]
    z = abs(est - truth) / se
    flag = "ok" if z < 3 else "** CHECK **"
    print(f"{name:6s}  PW = {est:10.5f} ± {se:.5f}   "
          f"analytic = {truth:10.5f}   z = {z:5.2f}   {flag}")

delta   PW =    0.63692 ± 0.00020   analytic =    0.63683   z =  0.43   ok
vega    PW =   37.56368 ± 0.06616   analytic =   37.52403   z =  0.60   ok


Delta's is 0.0002 / 0.637 ≈ 0.03%. Vega's is 0.066 / 37.5 ≈ 0.18%, six times noisier for the same path count. That's the antithetic asymmetry showing up empirically: Delta's integrand is near-symmetric in Z, so the antithetic pairing cancels hard, while Vega's $(\sqrt(T)Z - \sigma T)$ weight flips sign with Z and cancels far less. 

## Likelihood-ratio scores and estimators

The score is the $\partial_\theta$ of the lognormal log-density. Both are
payoff-agnostic - swap in any $f$:

$$\frac{\partial \log p}{\partial S_0} = \frac{Z}{S_0\,\sigma\sqrt{T}} \;\;\Rightarrow\;\; \Delta_{\text{LR}} = e^{-rT}\,\mathbb{E}\!\left[f(S_T)\,\frac{Z}{S_0\,\sigma\sqrt{T}}\right]$$

$$\frac{\partial \log p}{\partial \sigma} = \frac{Z^2 - 1}{\sigma} - Z\sqrt{T} \;\;\Rightarrow\;\; \mathcal{V}_{\text{LR}} = e^{-rT}\,\mathbb{E}\!\left[f(S_T)\left(\frac{Z^2 - 1}{\sigma} - Z\sqrt{T}\right)\right]$$

The vega score sits in *both* $m$ and $s$ (via $-\tfrac12\sigma^2 T$ and
$\sigma\sqrt{T}$) - missing either term gives a wrong-but-plausible vega.
Free correctness check: every score has mean zero
($\mathbb{E}[Z] = \mathbb{E}[Z^2 - 1] = 0$).

In [6]:
# --- canonical ATM test case for Likelihood ---
call_payoff = lambda ST: np.maximum(ST - 100.0, 0.0)
lr = bsm_greeks_lr(S, K, T, r, sigma, payoff=call_payoff, n_paths=1_000_000, seed=42)
delta_true, vega_true = analytic_delta_vega(S, K, T, r, sigma)

for name, truth in [("delta", delta_true), ("vega", vega_true)]:
    est, se = lr[name]
    z = abs(est - truth) / se
    flag = "ok" if z < 3 else "** CHECK **"
    print(f"{name:6s}  PW = {est:10.5f} ± {se:.5f}   "
          f"analytic = {truth:10.5f}   z = {z:5.2f}   {flag}")

delta   PW =    0.63795 ± 0.00133   analytic =    0.63683   z =  0.84   ok
vega    PW =   37.82659 ± 0.27643   analytic =   37.52403   z =  1.09   ok


## When PW fails and LR works

It fails under 2 main reasons: **Path-dependent Payoffs**, and Non-Lipschitz conditions of payoff (i.e we have **jumps** not kinks).

### The digital (cash-or-nothing call, pays \$1)

$$f(S_T) = \mathbf{1}_{\{S_T > K\}}, \qquad V = e^{-rT}\,\Phi(d_2), \qquad \Delta_{\text{digital}} = \frac{e^{-rT}\,\varphi(d_2)}{S_0\,\sigma\sqrt{T}}$$

The payoff *jumps* at $K$ - not Lipschitz - so PW's $f' = 0$ almost everywhere
and pathwise returns **exactly 0** (confidently wrong). LR is unaffected because
it never touches $f$. This is the case that forces LR.

In [7]:
# --- the validation truth: digital (AKA binary) call analytic Delta which is a jump ---
def digital_analytic_delta(S, K, T, r, sigma):
    """
    Cash-or-nothing call paying $1. V = e^{-rT} Phi(d2).
    Delta = e^{-rT} phi(d2) / (S sigma sqrt(T)).
    """
    d2 = (np.log(S / K) + (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return np.exp(-r * T) * norm.pdf(d2) / (S * sigma * np.sqrt(T))

# --- the payoff: a jump, not a kink ---
digital_payoff = lambda ST: (ST > 100.0).astype(float)

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_paths, seed = 1_000_000, 42

truth = digital_analytic_delta(S, K, T, r, sigma)

# --- METHOD 1: Likelihood-ratio (differentiates the density) ---
lr = bsm_greeks_lr(S, K, T, r, sigma, payoff=digital_payoff,
                   n_paths=n_paths, seed=seed)
lr_delta, lr_se = lr["delta"]
z_lr = abs(lr_delta - truth) / lr_se

# --- METHOD 2: Pathwise (differentiates the payoff) ---
# The pathwise delta integrand is  disc * f'(S_T) * dS_T/dS0.
# For the digital, f(x) = 1_{x>K} is flat everywhere it is differentiable,
# so f'(S_T) = 0 for every path. We build that integrand explicitly:
rng = np.random.default_rng(seed)
n_pairs = n_paths // 2
Z = rng.standard_normal(n_pairs)
Z_all = np.concatenate([Z, -Z])
S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_all)
disc = np.exp(-r * T)

f_prime = np.zeros_like(S_T)            # derivative of the indicator: 0 a.s.
pw_delta_paths = disc * f_prime * S_T / S
pw_delta = pw_delta_paths.mean()
pw_se = pw_delta_paths.std(ddof=1) / np.sqrt(len(pw_delta_paths))

# --- verdict ---
print(f"analytic truth         = {truth:10.6f}")
print(f"LR delta   = {lr_delta:10.6f} ± {lr_se:.6f}   z = {z_lr:5.2f}")
print(f"PW delta   = {pw_delta:10.6f} ± {pw_se:.6f}   <- the lie")

analytic truth         =   0.018762
LR delta   =   0.018772 ± 0.000021   z =  0.46
PW delta   =   0.000000 ± 0.000000   <- the lie


# Lessons: choosing a Monte Carlo Greek method

## Convergence - the rate is the same, only the constant differs

Every estimator here is a sample mean, so the CLT fixes the rate at

$$\text{SE} = \frac{\sigma_{\text{est}}}{\sqrt{N}} = O\!\left(N^{-1/2}\right)$$

for all of them. You *cannot* beat $N^{-1/2}$ by changing estimator - that needs
quasi-Monte Carlo (Sobol), which is a different machine. What changes between
methods is the constant $\sigma_{\text{est}}$, i.e. which curve of the same shape
you sit on. Variance reduction (antithetic, control variates) attacks the
constant, never the exponent.

## Method comparison

| Method | Rate | Bias? | Tuning | Variance constant |
|---|---|---|---|---|
| Pathwise | $N^{-1/2}$ | none | - | small (smooth payoffs only) |
| Likelihood-ratio | $N^{-1/2}$ | none | - | large; blows up as $T \to 0$ or $\sigma \to 0$ |
| Bump-and-revalue | worse than $N^{-1/2}$ | $O(h^2)$ | bump $h$ | $O(1/h)$, explodes as $h \to 0$ |

- **Pathwise** - unbiased, lowest variance, no tuning. *But* requires a Lipschitz
  payoff: works for vanillas (kink), fails silently for digitals/barriers (jump),
  returning a confidently wrong zero.
- **Likelihood-ratio** - unbiased, payoff-agnostic, handles any payoff including
  jumps. *But* noisier, and the score $\propto 1/(\sigma\sqrt{T})$ makes it
  pathological for short-dated or low-vol options.
- **Bump-and-revalue** (Week 1 FD) - the only one with *two* error sources: bias
  $O(h^2)$ from the finite bump plus variance $O(1/(h\sqrt{N}))$ that grows as
  $h \to 0$. Optimal $h$ trades them off; best rate is worse than $N^{-1/2}$.
  Common random numbers rescue the smooth case but fail on jumps, same as PW.

## Empirical variance ordering (this notebook)

LR is noisier than PW for both Greeks at equal $N$:

| Greek | PW SE | LR SE | ratio |
|---|---|---|---|
| Delta | 0.00020 | 0.00133 | ~6.5x |
| Vega | 0.06616 | 0.27643 | ~4.2x |

To match PW accuracy, LR delta needs $\sim 6.5^2 \approx 42\times$ the paths
(halving SE costs $4\times$ work under $N^{-1/2}$). LR vega is the
absolute-worst estimator (SE 0.276), as predicted.

## Antithetic variates - helps odd integrands, useless for even ones

Antithetic works by negative correlation under $Z \to -Z$, which needs the
integrand to respond to the *sign* of $Z$:

- **LR delta** (score $\propto Z$, odd): flips sign - strong cancellation, big win.
- **LR vega** (score has $(Z^2-1)/\sigma$, even): $Z^2$ is unchanged by the flip -
  antithetic does nothing for the dominant term. Vega barely improves.

Same reason PW delta antithetic-cancels hard (near-symmetric in $Z$) while PW
vega's $(\sqrt{T}Z - \sigma T)$ weight cancels less.

## The epistemic lesson - a tiny SE is a red flag, not a green light

The PW digital returned $0.000000 \pm 0.000000$: zero estimate, zero error bar.
Run it through the trusted z-score check and it gives $z = 0.018762 / 0 = \infty$
- the check itself breaks on the one case that most needs a warning.

**SE measures spread, not correctness.** It says how tightly an estimator
concentrates, nothing about *what* it concentrates on. The z-score guards against
noise and is completely blind to bias. A confidently-wrong estimator has a small
SE, and a small SE is exactly what we were trained to trust.

This bites hardest where there's no analytic truth to compare against - exactly
the models we built MC Greeks for (Heston, rBergomi). There, a PW digital's
$0 \pm 0$ would look like *spectacular convergence* and get reported as the most
precise number, when it is the most wrong.

**Defensive habit:** check the payoff is Lipschitz (kink, not jump) *before*
trusting the SE. When variance collapses toward zero, suspect validity before
celebrating precision. The error bar is necessary, never sufficient.